# 📤 cryoDRGN — export a particle subset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab_export_subset.ipynb)

Take a set of particle indices — from `cryodrgn filter`, a k-means cluster, an occupancy
percentile cut, UMAP lobes, anything — and write out **the subset and its complement** as a
new particle stack together with matching refinement metadata for cryoSPARC or RELION.

| Step | What happens |
|------|--------------|
| 1. Setup | Install cryoDRGN, mount Drive (a **CPU runtime is fine** — no GPU used) |
| 2. Inputs | Stack, `pose.pkl`, `ctf.pkl`, and the original `.cs`/`.star` if you have it |
| 3. Selection | Load indices; choose selection, complement, or both |
| 4. Export | `.mrcs` + `pose.pkl` + `ctf.pkl` + `.star` and/or `.cs` per set |
| 5. Verify | Cross-check counts, box size, pixel size and a per-row spot check |
| 6. Bundle | Zip to Drive |

### Two kinds of output, and they are not interchangeable

- **Filtered original `.cs`/`.star`** — every field of your original metadata, subset by row.
  It still points at your **full-resolution** raw particles, so this is what you re-import into
  cryoSPARC or RELION to carry on refining at full res. Requires the original file.
- **New `.mrcs` + generated `.star`/`.cs`** — a self-contained bundle at whatever box size your
  cryoDRGN stack uses. Portable, but downsampled.

This notebook writes both when it can.

> **Indices are 0-based positions into the particle stack**, in the same order as the `.cs`/`.star`
> the poses and CTF were parsed from. That holds if the stack was built from the full metadata file
> in order. If you trained with `--ind`, indices taken from `z.N.pkl` are positions *within that
> subset* — compose them first (`cryodrgn_utils select_clusters --parent-ind`).

## 1 · Setup

In [ ]:
#@title 1.1 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# --- realign torchvision with torch -------------------------------------------------
# cryoDRGN pins torch<2.10, so pip may DOWNGRADE Colab's torch. Colab's pre-installed
# torchvision was compiled against the newer torch, and once they disagree importing it
# raises "operator torchvision::nms does not exist". That breaks EVERY cryodrgn command,
# because the CLI eagerly imports all command modules and analyze_landscape_full imports
# umap -> torchvision. Matching pair is torch 2.N <-> torchvision 0.(N+15).
import importlib.metadata as md

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

tver, tvver = _ver("torch"), _ver("torchvision")
if tver and tvver:
    tmaj, tmin = (int(x) for x in tver.split(".")[:2])
    tvmin = int(tvver.split(".")[1])
    want = tmin + 15 if tmaj == 2 else None
    if want is not None and tvmin != want:
        print(f"\n⚠️  torch {tver} and torchvision {tvver} are incompatible "
              f"(cryoDRGN's torch<2.10 pin downgraded torch).")
        print(f"   Installing torchvision 0.{want}.* to match...")
        fix = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             f"torchvision==0.{want}.*"])
        if fix.returncode == 0:
            print(f"   ✅ torchvision realigned to 0.{want}.*")
        else:
            print(f"   ❌ Could not install torchvision 0.{want}.* — if cryodrgn commands "
                  f"fail with 'torchvision::nms does not exist', run:")
            print(f"      !pip install --no-deps 'torchvision==0.{want}.*'")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.2 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted. The `cryodrgn --version` smoke-test is
#@markdown the important one: the CLI imports *every* command module on startup, so a broken
#@markdown dependency anywhere makes all commands fail — better to catch it here than mid-run.
import sys, subprocess
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — fine for Step 4 (CPU-only), but Step 6 needs a GPU (cell 1.1).")

# torchvision must match torch or `import umap` blows up inside the cryodrgn CLI
try:
    import torchvision
    print(f"torchvision      : {torchvision.__version__} (ok)")
except Exception as e:
    msg = str(e).splitlines()[0]
    want = "0.%d.*" % (int(torch.__version__.split(".")[1]) + 15)
    if "numpy.dtype size changed" in msg or "binary incompatibility" in msg:
        # cryoDRGN pins numpy<1.27, downgrading Colab's numpy 2.x. C extensions that were
        # compiled against numpy 2.x headers then fail their ABI check on import.
        print(f"⚠️  torchvision fails a NumPy ABI check: {msg}")
        print("   Cause: cryoDRGN pins numpy<1.27, so Colab's numpy 2.x was downgraded and")
        print("   torchvision (built against numpy 2.x) no longer matches.")
        print("   This is USUALLY HARMLESS: cryoDRGN never imports torchvision itself — only")
        print("   umap does, and cryodrgn.analysis imports umap lazily. Every cryodrgn command")
        print("   also runs as a subprocess. Treat the CLI check below as the real verdict, and")
        print("   don't 'fix' this unless something actually fails.")
    else:
        print(f"❌ torchvision is broken: {msg}")
        print("   Looks like a torch/torchvision version mismatch rather than a NumPy issue.")
        print(f"   Fix with:  !pip install --no-deps 'torchvision=={want}'")
        print("   then re-run this cell (no restart needed).")

print("\n$ cryodrgn --version")
r = subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-2000:])
if r.returncode != 0:
    raise RuntimeError("The cryodrgn CLI failed to start — fix the error above before continuing.")
print("\n✅ CLI healthy — all command modules import cleanly.")

In [ ]:
#@title 1.3 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

## 2 · Inputs

The stack, poses and CTF you used with cryoDRGN, plus — if you have them — the **original**
cryoSPARC `.cs` or RELION `.star`. Supplying the original is what makes a faithful
full-resolution export possible; without it the notebook can still synthesise metadata from
`pose.pkl` + `ctf.pkl`, but only describing the downsampled stack.

> `cryodrgn_utils filter_cs` requires a `.cs` input (`filter_cs.py:31-33`) and `write_star`
> accepts only `.mrcs`/`.txt`/`.star` (`write_star.py:84-86`) — so a `.cs` cannot be turned
> directly into a `.star`. The notebook routes around that via the filtered `.mrcs`.

In [ ]:
#@title 2.1 · Locate the inputs { display-mode: "form" }
#@markdown Drive project folder — the same one the main notebook used.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown Local scratch for the outputs before they are copied to Drive.
local_work_dir = "/content/cryodrgn_export"  #@param {type:"string"}
#@markdown Particle stack cryoDRGN used (blank = auto-detect a `*.mrcs`/`*.txt` in the project).
particles = ""  #@param {type:"string"}
#@markdown Poses / CTF (blank = `pose.pkl` / `ctf.pkl` in the project folder).
pose_pkl = ""  #@param {type:"string"}
ctf_pkl = ""  #@param {type:"string"}
#@markdown **Original** cryoSPARC `.cs` — enables a full-resolution `.cs` export. Optional.
original_cs = ""  #@param {type:"string"}
#@markdown **Original** RELION `.star` — enables a full-resolution `.star` export. Optional.
original_star = ""  #@param {type:"string"}

import os, glob
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource

DRIVE_DIR = os.path.abspath(drive_project_dir)
WORK_DIR = os.path.abspath(local_work_dir)
os.makedirs(WORK_DIR, exist_ok=True)
if not os.path.isdir(DRIVE_DIR):
    raise FileNotFoundError(f"{DRIVE_DIR} not found — check drive_project_dir (and run 1.3).")

def _pick(explicit, *patterns):
    if explicit.strip():
        p = os.path.abspath(explicit.strip())
        if not os.path.exists(p):
            raise FileNotFoundError(p)
        return p
    for pat in patterns:
        hits = sorted(glob.glob(os.path.join(DRIVE_DIR, pat)))
        hits += sorted(glob.glob(os.path.join(DRIVE_DIR, "*", pat)))
        if hits:
            return hits[0]
    return ""

parts = _pick(particles, "*.mrcs", "*.txt")
pose = _pick(pose_pkl, "pose.pkl")
ctf = _pick(ctf_pkl, "ctf.pkl")
ocs = _pick(original_cs) if original_cs.strip() else ""
ostar = _pick(original_star) if original_star.strip() else ""

missing = [n for n, p in (("particles", parts), ("pose.pkl", pose), ("ctf.pkl", ctf)) if not p]
if missing:
    raise FileNotFoundError(f"Could not locate: {', '.join(missing)} — set them explicitly above.")

src = ImageSource.from_file(parts, lazy=True)
N, stack_D = src.n, src.D
ctf9 = utils.load_pkl(ctf)
if ctf9.ndim != 2 or ctf9.shape[1] != 9:
    raise ValueError(f"{ctf} has shape {ctf9.shape}; expected (N, 9).")
rot, trans = utils.load_pkl(pose)
for nm, ln in (("ctf.pkl", len(ctf9)), ("pose.pkl", len(rot))):
    if ln != N:
        raise ValueError(f"{nm} has {ln:,} rows but the stack has {N:,} particles.")

ctf_box, ctf_apix = float(ctf9[0, 0]), float(ctf9[0, 1])
cur_apix = round(ctf_apix * ctf_box / stack_D, 6)

for k, v in dict(EX_DRIVE=DRIVE_DIR, EX_WORK=WORK_DIR, EX_PARTS=parts, EX_POSE=pose,
                 EX_CTF=ctf, EX_CS=ocs, EX_STAR=ostar, EX_N=str(N),
                 EX_D=str(stack_D), EX_APIX=str(cur_apix)).items():
    os.environ[k] = v
os.chdir(WORK_DIR)

print(f"📁 project        : {DRIVE_DIR}")
print(f"🧊 stack          : {parts}")
print(f"                    {N:,} particles at {stack_D}x{stack_D}")
print(f"📐 poses          : {pose}")
print(f"🔬 ctf            : {ctf}")
print(f"                    records box {int(ctf_box)} at {ctf_apix} Å/px")
print(f"🗂  original .cs   : {ocs or '(none — no full-resolution .cs export)'}")
print(f"🗂  original .star : {ostar or '(none — no full-resolution .star export)'}")
if int(ctf_box) != stack_D:
    print(f"\nℹ️  The stack is downsampled {int(ctf_box)} → {stack_D}, so the exported metadata")
    print(f"    needs box {stack_D} at {cur_apix} Å/px. Cell 4.1 rewrites columns 0-1 of the CTF")
    print(f"    accordingly — write_star would otherwise copy {int(ctf_box)} / {ctf_apix} straight")
    print(f"    into _rlnImageSize / _rlnImagePixelSize while writing {stack_D}px images.")

## 3 · The selection

Point at a `.pkl` (a NumPy integer array or boolean mask), a `.txt` of one index per line, or a
folder/zip of several. For each one you can export the **selection**, its **complement**, or both.

In [ ]:
#@title 3.1 · Load indices and form the sets to export { display-mode: "form" }
#@markdown A `.pkl`/`.txt` index file, or a folder or `.zip` containing several.
indices_source = ""  #@param {type:"string"}
#@markdown What to write out for each input index set.
export = "both"  #@param ["selection", "complement", "both"]
#@markdown Optional glob to keep only some sets when pointing at a folder, e.g. `*lobe01*`.
name_filter = "*"  #@param {type:"string"}
#@markdown Treat all the input sets together as one selection (union) instead of separately.
merge_into_one = False  #@param {type:"boolean"}

import os, re, glob, zipfile, fnmatch, json
import numpy as np
from cryodrgn import utils

N = int(os.environ["EX_N"])
WORK_DIR = os.environ["EX_WORK"]
src = indices_source.strip()
if not src or not os.path.exists(src):
    raise FileNotFoundError("Set indices_source to a .pkl/.txt file, or a folder/zip of them.")

if src.lower().endswith(".zip"):
    dest = os.path.join(WORK_DIR, "index_input")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(src) as zf:
        zf.extractall(dest)
    src = dest
    print(f"unzipped → {dest}")

if os.path.isfile(src):
    files = [src]
else:
    files = []
    for root, dirs, fns in os.walk(src):
        dirs[:] = [d for d in dirs if d != "__MACOSX"]
        files += [os.path.join(root, f) for f in fns
                  if not f.startswith("._") and f.lower().endswith((".pkl", ".txt"))]

by_stem = {}
for p in sorted(files):
    stem = re.sub(r"^(ind_|indices_|particles_)", "", os.path.splitext(os.path.basename(p))[0])
    if stem not in by_stem or p.lower().endswith(".pkl"):
        by_stem[stem] = p
by_stem = {k: v for k, v in by_stem.items() if fnmatch.fnmatch(k, name_filter)}
if not by_stem:
    raise FileNotFoundError(f"No .pkl/.txt index files found under {src} matching '{name_filter}'.")

def _load(p):
    a = utils.load_pkl(p) if p.lower().endswith(".pkl") else np.loadtxt(p, dtype=np.int64, ndmin=1)
    a = np.asarray(a).ravel()
    if a.dtype == bool:                       # a boolean mask is also acceptable
        if a.size != N:
            raise ValueError(f"{os.path.basename(p)}: boolean mask of {a.size}, expected {N}")
        a = np.where(a)[0]
    a = a.astype(np.int64)
    if a.size == 0:
        raise ValueError(f"{os.path.basename(p)} is empty")
    if a.min() < 0 or a.max() >= N:
        raise ValueError(f"{os.path.basename(p)}: index range {a.min()}..{a.max()} "
                         f"outside the stack's 0..{N-1}")
    if np.unique(a).size != a.size:
        raise ValueError(f"{os.path.basename(p)} contains duplicates")
    return np.unique(a)

loaded = {k: _load(v) for k, v in sorted(by_stem.items())}
if merge_into_one:
    merged = np.unique(np.concatenate(list(loaded.values())))
    print(f"merged {len(loaded)} input set(s) → one selection of {merged.size:,}")
    loaded = {"selection": merged}

out_dir = os.path.join(WORK_DIR, "index_sets")
os.makedirs(out_dir, exist_ok=True)
sets = {}
for stem, idx in loaded.items():
    comp = np.setdiff1d(np.arange(N, dtype=np.int64), idx, assume_unique=True)
    if export in ("selection", "both"):
        f = os.path.join(out_dir, f"{stem}.pkl")
        utils.save_pkl(idx, f)
        sets[stem] = f
    if export in ("complement", "both"):
        if comp.size == 0:
            print(f"⚠️  {stem}: complement is empty — skipping it")
        else:
            f = os.path.join(out_dir, f"{stem}_complement.pkl")
            utils.save_pkl(comp, f)
            sets[f"{stem}_complement"] = f

w = max(len(k) for k in sets)
print(f"\n{'set to export'.ljust(w)}  {'N':>9}   {'% of stack':>10}")
print("-" * (w + 26))
for k, f in sorted(sets.items()):
    n = len(np.asarray(utils.load_pkl(f)))
    print(f"{k.ljust(w)}  {n:>9,}   {n/N*100:>9.2f}%")
print("-" * (w + 26))
print(f"{'stack total'.ljust(w)}  {N:>9,}")
for stem, idx in loaded.items():
    if export == "both":
        assert idx.size + (N - idx.size) == N
        print(f"\n{stem}: selection + complement = {idx.size:,} + {N-idx.size:,} = {N:,} ✓")

os.environ["EX_SETS"] = json.dumps(sets)

## 4 · Export

Per set, using cryoDRGN's own utilities so nothing is reimplemented:

| Output | Command | Notes |
|---|---|---|
| `<set>.mrcs` | `filter_mrcs` | the subset stack, at the cryoDRGN box size |
| `<set>_pose.pkl`, `<set>_ctf.pkl` | `filter_pkl` | so the subset can be retrained in cryoDRGN |
| `<set>.star` | `write_star` | built from the filtered stack + filtered CTF/poses |
| `<set>_fullres.cs` | `filter_cs` | the **original** `.cs` subset by row — full resolution |
| `<set>_fullres.star` | `write_star --ind` | the **original** `.star` subset by row — full resolution |

**The CTF box-size correction.** `write_star` writes CTF columns 0 and 1 into the optics table as
`_rlnImageSize` and `_rlnImagePixelSize` (`write_star.py:34-44, 198-206`), but derives the shifts
from the *stack's* box: `D = particles[0].shape[-1]; trans = poses[1] * D` (`:185-187`). Feed it a
downsampled stack with an unmodified `ctf.pkl` and you get a `.star` claiming a 400 px box at
0.83 Å/px while carrying 128 px images and 128 px shifts. The cell rewrites those two columns to
match the stack — the same arithmetic `ctf.py:160-162` uses internally.

In [ ]:
#@title 4.1 · Write the stacks and metadata { display-mode: "form" }
#@markdown Write a RELION `.star` alongside each subset stack.
write_star_files = True  #@param {type:"boolean"}
#@markdown Write a full-resolution `.cs` / `.star` by filtering the original (needs one in 2.1).
write_fullres = True  #@param {type:"boolean"}
#@markdown Absolute rather than relative particle paths inside the generated `.star`.
full_path = False  #@param {type:"boolean"}
#@markdown Recreate outputs that already exist.
overwrite = False  #@param {type:"boolean"}

import os, json, shutil, subprocess, time
import numpy as np
from cryodrgn import utils

sets = json.loads(os.environ["EX_SETS"])
PARTS, POSE, CTF = (os.environ[k] for k in ("EX_PARTS", "EX_POSE", "EX_CTF"))
OCS, OSTAR = os.environ["EX_CS"], os.environ["EX_STAR"]
WORK, DRIVE = os.environ["EX_WORK"], os.environ["EX_DRIVE"]
stack_D, cur_apix = int(os.environ["EX_D"]), float(os.environ["EX_APIX"])

out_root = os.path.join(WORK, "subsets")
os.makedirs(out_root, exist_ok=True)
os.environ["EX_OUT"] = out_root

def run(cmd):
    print("   $", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError(f"failed (exit {r.returncode}): {' '.join(cmd[:3])} ...")

for name, ind_pkl in sorted(sets.items()):
    d = os.path.join(out_root, name)
    os.makedirs(d, exist_ok=True)
    mrcs = os.path.join(d, f"{name}.mrcs")
    pose_out = os.path.join(d, f"{name}_pose.pkl")
    ctf_out = os.path.join(d, f"{name}_ctf.pkl")
    star_out = os.path.join(d, f"{name}.star")
    n_sel = len(np.asarray(utils.load_pkl(ind_pkl)))

    print(f"\n=== {name}  ({n_sel:,} particles) " + "=" * 24)
    if os.path.exists(mrcs) and not overwrite:
        print("   stack exists — skipping (tick overwrite to redo)")
    else:
        t0 = time.time()
        run(["cryodrgn_utils", "filter_mrcs", PARTS, "--ind", ind_pkl, "-o", mrcs])
        print(f"   stack: {os.path.getsize(mrcs)/1e9:.2f} GB in {time.time()-t0:.0f}s")

    run(["cryodrgn_utils", "filter_pkl", POSE, "--ind", ind_pkl, "-o", pose_out])

    # Filter the CTF, then rewrite the box size / pixel size to describe the stack we just
    # wrote. Same arithmetic as ctf.load_ctf_for_training (ctf.py:160-162); without it the
    # .star optics table would describe the ORIGINAL box while the images are downsampled.
    c = np.asarray(utils.load_pkl(CTF))[np.asarray(utils.load_pkl(ind_pkl))].copy()
    if int(c[0, 0]) != stack_D:
        print(f"   ctf: rewriting box {int(c[0,0])} → {stack_D}, "
              f"{c[0,1]:.4f} → {cur_apix} Å/px")
        c[:, 1] = cur_apix
        c[:, 0] = stack_D
    utils.save_pkl(c, ctf_out)

    if write_star_files:
        if os.path.exists(star_out) and not overwrite:
            print("   .star exists — skipping")
        else:
            cmd = ["cryodrgn_utils", "write_star", mrcs, "-o", star_out,
                   "--ctf", ctf_out, "--poses", pose_out]
            if full_path:
                cmd.append("--full-path")
            run(cmd)

    if write_fullres and OCS:
        cs_out = os.path.join(d, f"{name}_fullres.cs")
        if os.path.exists(cs_out) and not overwrite:
            print("   full-res .cs exists — skipping")
        else:
            run(["cryodrgn_utils", "filter_cs", OCS, "--ind", ind_pkl, "-o", cs_out])
    if write_fullres and OSTAR:
        fs_out = os.path.join(d, f"{name}_fullres.star")
        if os.path.exists(fs_out) and not overwrite:
            print("   full-res .star exists — skipping")
        else:
            run(["cryodrgn_utils", "write_star", OSTAR, "-o", fs_out, "--ind", ind_pkl])

    for f in sorted(os.listdir(d)):
        print(f"   → {f:<38} {os.path.getsize(os.path.join(d, f))/1e6:>10,.1f} MB")

print("\n" + "=" * 60 + f"\n✅ {len(sets)} subset(s) written to {out_root}")

## 5 · Verify

Every claim the exports make is checked against the original: row counts, the pixel data itself
(a random sample of rows must be byte-identical to the source), the CTF's defocus columns, the
rewritten box size and pixel size, and the `.star` optics table. It raises rather than warns.

In [ ]:
#@title 5.1 · Verify the exports { display-mode: "form" }
#@markdown Re-reads every output and checks it against the original. Cheap; always worth running.
spot_checks = 5  #@param {type:"integer"}

import os, json, glob
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource
from cryodrgn.starfile import parse_star

sets = json.loads(os.environ["EX_SETS"])
out_root, PARTS, CTF = os.environ["EX_OUT"], os.environ["EX_PARTS"], os.environ["EX_CTF"]
stack_D, cur_apix = int(os.environ["EX_D"]), float(os.environ["EX_APIX"])
full_src = ImageSource.from_file(PARTS, lazy=True)
full_ctf = np.asarray(utils.load_pkl(CTF))
rng = np.random.default_rng(0)
problems = []

for name, ind_pkl in sorted(sets.items()):
    d = os.path.join(out_root, name)
    ind = np.asarray(utils.load_pkl(ind_pkl))
    print(f"\n=== {name} ===")

    mrcs = os.path.join(d, f"{name}.mrcs")
    sub = ImageSource.from_file(mrcs, lazy=True)
    ok = sub.n == len(ind) and sub.D == stack_D
    print(f"  stack      : {sub.n:,} images at {sub.D}x{sub.D}   "
          f"{'✓' if ok else '✗ expected %d at %d' % (len(ind), stack_D)}")
    problems += [] if ok else [f"{name}: stack shape"]

    # pixel-exact spot check: row k of the subset must equal row ind[k] of the original
    k = rng.choice(len(ind), size=min(int(spot_checks), len(ind)), replace=False)
    bad = [int(i) for i in k
           if not np.allclose(np.asarray(sub.images(np.array([i]))),
                              np.asarray(full_src.images(np.array([ind[i]]))))]
    print(f"  images     : {len(k)} random row(s) match the original   {'✓' if not bad else '✗ ' + str(bad)}")
    problems += [] if not bad else [f"{name}: image mismatch at {bad}"]

    c = np.asarray(utils.load_pkl(os.path.join(d, f"{name}_ctf.pkl")))
    same_defocus = np.allclose(c[:, 2:], full_ctf[ind][:, 2:])
    box_ok = int(c[0, 0]) == stack_D and abs(c[0, 1] - cur_apix) < 1e-4
    print(f"  ctf        : {len(c):,} rows, box {int(c[0,0])} at {c[0,1]:.4f} Å/px   "
          f"{'✓' if box_ok else '✗'}   defocus preserved {'✓' if same_defocus else '✗'}")
    problems += ([] if box_ok else [f"{name}: ctf box"]) + \
                ([] if same_defocus else [f"{name}: ctf defocus"])

    r, t = utils.load_pkl(os.path.join(d, f"{name}_pose.pkl"))
    print(f"  poses      : {len(r):,} rotations, {len(t):,} translations   "
          f"{'✓' if len(r) == len(ind) == len(t) else '✗'}")
    problems += [] if len(r) == len(ind) == len(t) else [f"{name}: pose count"]

    star = os.path.join(d, f"{name}.star")
    if os.path.exists(star):
        df, optics = parse_star(star)
        n_ok = len(df) == len(ind)
        msg = f"  .star      : {len(df):,} rows {'✓' if n_ok else '✗'}"
        if optics is not None and "_rlnImagePixelSize" in optics:
            apix_star = float(optics["_rlnImagePixelSize"].iloc[0])
            size_star = int(float(optics["_rlnImageSize"].iloc[0]))
            a_ok = abs(apix_star - cur_apix) < 1e-4 and size_star == stack_D
            msg += f", optics {size_star}px at {apix_star:.4f} Å/px {'✓' if a_ok else '✗'}"
            problems += [] if a_ok else [f"{name}: star optics"]
        print(msg)
        problems += [] if n_ok else [f"{name}: star rows"]

    for tag in ("_fullres.cs", "_fullres.star"):
        f = os.path.join(d, name + tag)
        if os.path.exists(f):
            n = len(np.load(f)) if tag.endswith(".cs") else len(parse_star(f)[0])
            print(f"  {tag:<11}: {n:,} rows   {'✓' if n == len(ind) else '✗'}")
            problems += [] if n == len(ind) else [f"{name}{tag}: rows"]

print("\n" + "=" * 60)
if problems:
    raise RuntimeError("Verification failed:\n  - " + "\n  - ".join(problems))
print("✅ All exports verified: counts, pixel data, CTF and box size all consistent.")

## 6 · Bundle

In [ ]:
#@title 6.1 · Copy to Drive and (optionally) zip { display-mode: "form" }
#@markdown Metadata files are small; the `.mrcs` stacks are not. Untick to copy metadata only.
include_stacks = True  #@param {type:"boolean"}
#@markdown Also build a single zip of the metadata for downloading.
zip_metadata = True  #@param {type:"boolean"}

import os, shutil, glob
out_root, DRIVE, WORK = os.environ["EX_OUT"], os.environ["EX_DRIVE"], os.environ["EX_WORK"]
dst_root = os.path.join(DRIVE, "subsets")
os.makedirs(dst_root, exist_ok=True)

meta_ext = (".pkl", ".star", ".cs")
copied = tot = 0
for d in sorted(glob.glob(os.path.join(out_root, "*"))):
    if not os.path.isdir(d):
        continue
    dst = os.path.join(dst_root, os.path.basename(d))
    os.makedirs(dst, exist_ok=True)
    for f in sorted(os.listdir(d)):
        if f.endswith(".mrcs") and not include_stacks:
            continue
        s, t = os.path.join(d, f), os.path.join(dst, f)
        if os.path.exists(t) and os.path.getsize(t) == os.path.getsize(s):
            continue
        shutil.copyfile(s, t)          # not copy2: Drive's FUSE mount dislikes metadata copies
        copied += 1
        tot += os.path.getsize(s)
print(f"✅ {copied} file(s), {tot/1e9:.2f} GB → {dst_root}")

if zip_metadata:
    stage = os.path.join(WORK, "_meta")
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    for d in sorted(glob.glob(os.path.join(out_root, "*"))):
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.endswith(meta_ext):
                shutil.copyfile(os.path.join(d, f), os.path.join(stage, f))
    base = os.path.join(WORK, "subset_metadata")
    shutil.make_archive(base, "zip", stage)
    shutil.copyfile(base + ".zip", os.path.join(DRIVE, "subset_metadata.zip"))
    print(f"✅ metadata zip ({os.path.getsize(base + '.zip')/1e6:.1f} MB) → "
          f"{DRIVE}/subset_metadata.zip")
    try:
        from google.colab import files
        files.download(base + ".zip")
    except Exception as e:
        print(f"   (browser download unavailable: {e})")

---
### Taking the outputs onward

**Back into cryoSPARC at full resolution.** Use `<set>_fullres.cs`. It is your original `.cs`
subset by row, so every field — micrograph paths, full-size blob references, existing
`alignments3D` — is intact and it still points at the full-resolution particles. Import it with
*Import Result Group*, or place it where a job expects its `particles.cs`.

**Back into RELION at full resolution.** Use `<set>_fullres.star`, likewise a row subset of your
original.

**Anywhere else, self-contained.** `<set>.mrcs` + `<set>.star` travel together and describe the
**downsampled** box. Keep them in the same folder unless you exported with `full_path`.

**Back into cryoDRGN.** `<set>.mrcs` + `<set>_pose.pkl` + `<set>_ctf.pkl` drop straight into
`train_vae`. Because the subset is already applied, you do **not** pass `--ind` — which also means
`--shuffler-size` stays available for lazy loading, since the shuffler refuses to run alongside
`--ind` (`dataset.py:395-403`).

### What is not carried over

The generated `.star` describes what cryoDRGN knows: image name, defocus, optics, and poses if you
supplied them. Micrograph names, coordinates, per-particle scale factors, figure-of-merit columns
and anything else in your original metadata survive only in the `_fullres` outputs. If you need
those, filter the original — that is what `write_fullres` is for.